# K-Means Clustering

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

K-means partitions observations into k clusters by minimizing within-cluster squared distances.

**Highest-probability exam pattern:** Standardize variables, make an elbow plot, choose k, fit k-means, and interpret clusters cautiously.

## Assumptions, intuition, and theory

K-means works best for roughly spherical, similarly sized clusters. It is distance-based and sensitive to scaling, random starts, and outliers. Labels are arbitrary names, not ground truth unless simulation labels exist.

## Python method notes and documentation

`KMeans.fit_predict` returns cluster assignments, `inertia_` is within-cluster sum of squares, `n_init` controls random restarts, and silhouette score summarizes separation when k is at least 2.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [silhouette_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
- [adjusted_rand_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html)
- [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)
- [make_blobs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for elbow summaries.
import matplotlib.pyplot as plt  # Import plotting tools for elbow plots.
from sklearn.cluster import KMeans  # Import k-means clustering.
from sklearn.datasets import make_blobs  # Import cluster simulator.
from sklearn.metrics import adjusted_rand_score, silhouette_score  # Import clustering quality summaries.
from sklearn.preprocessing import StandardScaler  # Import standardization before distance-based clustering.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_cluster_data(n=180, centers=3, cluster_std=0.8, random_state=123):  # Define a labeled cluster simulator for recovery checks.
    X, true_labels = make_blobs(  # Generate blob-shaped clusters with known labels.
        n_samples=n,  # Set the number of observations.
        centers=centers,  # Set the number of true clusters.
        cluster_std=cluster_std,  # Control cluster overlap.
        random_state=random_state,  # Fix the simulation seed.
    )  # Finish the simulator call.
    return X, true_labels  # Return data and true labels.


In [ ]:
X, true_labels = make_cluster_data(n=180, centers=3, cluster_std=0.8, random_state=RANDOM_SEED)  # Simulate clustered data with known labels.
X_scaled = StandardScaler().fit_transform(X)  # Standardize before distance-based clustering.
rows = []  # Create a list for k-means tuning results.
for k in range(1, 8):  # Try a range of possible cluster counts.
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_SEED)  # Create k-means with multiple random starts.
    labels = model.fit_predict(X_scaled)  # Fit k-means and return cluster assignments.
    silhouette = np.nan if k == 1 else silhouette_score(X_scaled, labels)  # Compute silhouette only when k is at least 2.
    rows.append({'k': k, 'inertia': model.inertia_, 'silhouette': silhouette})  # Store elbow and silhouette summaries.
results = pd.DataFrame(rows)  # Convert tuning results to a table.
final_model = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_SEED)  # Create final k-means model using the true cluster count for demonstration.
final_labels = final_model.fit_predict(X_scaled)  # Fit final clustering and get assignments.
ari = adjusted_rand_score(true_labels, final_labels)  # Compare recovered labels to true simulation labels.
display(results)  # Display the elbow and silhouette table.
print(f'Adjusted Rand index for k=3: {ari:.3f}')  # Print cluster recovery score.
plt.figure(figsize=(6, 4))  # Create an elbow-plot figure.
plt.plot(results['k'], results['inertia'], marker='o')  # Plot within-cluster sum of squares against k.
plt.xlabel('number of clusters k')  # Label the k axis.
plt.ylabel('inertia')  # Label the inertia axis.
plt.title('K-means elbow plot')  # Title the elbow plot.
plt.show()  # Render the elbow plot.
plt.figure(figsize=(6, 4))  # Create a cluster scatterplot figure.
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=final_labels, s=30, alpha=0.75)  # Plot standardized data colored by recovered cluster.
plt.xlabel('scaled x1')  # Label the first axis.
plt.ylabel('scaled x2')  # Label the second axis.
plt.title('K-means recovered clusters')  # Title the cluster plot.
plt.show()  # Render the cluster plot.


## Exam-style problem prompt

A prompt asks for clustering without labels. Standardize, compute k-means for several k values, draw an elbow plot, and describe the selected clusters.

## Adaptable code pattern

```python
# Step 1: Standardize numeric variables.
# Step 2: Run KMeans for a range of k values.
# Step 3: Record inertia and optionally silhouette score.
# Step 4: Use elbow reasoning to choose k.
# Step 5: Fit final k-means and plot clusters.
# Step 6: Interpret clusters as exploratory groups, not confirmed classes.
```

## What to remember for the exam

K-means is unsupervised. Do not talk about prediction accuracy unless the data were simulated with known labels for recovery checking.